# Tutorial 1: Learn about `requests`

Follow along with [this tutorial on RealPython](https://realpython.com/python-requests/). 
Complete the first 5 sections thoroughly (up to but not including User Other HTTP methods). Then jump to section Improve Performance to learn about some advanced tricks that you might need at some point when scraping websites that have a lot of content.

Use markdown headings as appropriate to enumerate sections.

In [4]:
python -m pip install requests

SyntaxError: invalid syntax (591647450.py, line 1)

## Inspect the response 

A Response is the object that contains the results of your request. Try making that same request again, but this time store the return value in a variable so you can get a closer look at its attributes and behaviors:


In [5]:
import requests
response = requests.get("https://api.github.com")

In this example, you’ve captured the return value of requests.get(). It’s an instance of Response, and you stored it in a variable called response. You can now use response to see a lot of information about the results of your GET request

## Work With Status Codes

A status code informs you of the status of the request.

a 200 OK status means that your request was successful, while a 404 NOT FOUND status means that the resource you were looking for wasn’t found.

In [6]:
response.status_code

200

Sometimes, you might want to use this information to make decisions in your code:

In [7]:
if response.status_code == 200:
    print("Success!")
elif response.status_code == 404:
    print("Not Found.")

Success!


Requests goes one step further in simplifying this process for you. If you use a Response instance in a Boolean context, such as a conditional statement, then it’ll evaluate to True when the status code is less than 400, and False otherwise.

That means you can modify the last example by rewriting the if statement:



In [8]:
if response:
    print("Success!")
else:
    raise Exception(f"Non-success status code: {response.status_code}")

Success!


n the code snippet above, you implicitly check whether the .status_code of response is between 200 and 399. If it’s not, then you raise an exception with an error message that includes the non-success status code wrapped in an f-string.

Keep in mind that this method does not verify whether the status code is equal to 200. This is because other status codes within the 200 to 399 range, such as 204 NO CONTENT and 304 NOT MODIFIED, are also considered successful because they provide some workable response.

In [9]:
import requests
from requests.exceptions import HTTPError

URLS = ["https://api.github.com", "https://api.github.com/invalid"]

for url in URLS:
    try:
        response = requests.get(url)
        response.raise_for_status()
    except HTTPError as http_err:
        print(f"HTTP error occurred: {http_err}")
    except Exception as err:
        print(f"Other error occurred: {err}")
    else:
        print("Success!")

Success!
HTTP error occurred: 404 Client Error: Not Found for url: https://api.github.com/invalid


## Access the Response Content

The response of a GET request often has some valuable information, known as a payload, in the message body. Using the attributes and methods of Response, you can view the payload in a variety of formats.

To see the response’s content in bytes, you use .content:


In [10]:
import requests

response = requests.get("https://api.github.com")
response.content


type(response.content)

bytes

While .content gives you access to the raw bytes of the response payload, you’ll often want to convert them into a string using a character encoding such as UTF-8. response will do that for you when you access .text:

In [11]:
response.text


type(response.text)

str

Because the decoding of bytes to a str requires an encoding scheme, Requests will try to guess the encoding based on the response’s headers if you don’t specify one. You can provide an explicit encoding by setting .encoding before accessing .text:

In [12]:
response.encoding = "utf-8"  # Optional: Requests infers this.
response.text

'{\n  "current_user_url": "https://api.github.com/user",\n  "current_user_authorizations_html_url": "https://github.com/settings/connections/applications{/client_id}",\n  "authorizations_url": "https://api.github.com/authorizations",\n  "code_search_url": "https://api.github.com/search/code?q={query}{&page,per_page,sort,order}",\n  "commit_search_url": "https://api.github.com/search/commits?q={query}{&page,per_page,sort,order}",\n  "emails_url": "https://api.github.com/user/emails",\n  "emojis_url": "https://api.github.com/emojis",\n  "events_url": "https://api.github.com/events",\n  "feeds_url": "https://api.github.com/feeds",\n  "followers_url": "https://api.github.com/user/followers",\n  "following_url": "https://api.github.com/user/following{/target}",\n  "gists_url": "https://api.github.com/gists{/gist_id}",\n  "hub_url": "https://api.github.com/hub",\n  "issue_search_url": "https://api.github.com/search/issues?q={query}{&page,per_page,sort,order}",\n  "issues_url": "https://api.g

In [13]:
response.json()

{'current_user_url': 'https://api.github.com/user',
 'current_user_authorizations_html_url': 'https://github.com/settings/connections/applications{/client_id}',
 'authorizations_url': 'https://api.github.com/authorizations',
 'code_search_url': 'https://api.github.com/search/code?q={query}{&page,per_page,sort,order}',
 'commit_search_url': 'https://api.github.com/search/commits?q={query}{&page,per_page,sort,order}',
 'emails_url': 'https://api.github.com/user/emails',
 'emojis_url': 'https://api.github.com/emojis',
 'events_url': 'https://api.github.com/events',
 'feeds_url': 'https://api.github.com/feeds',
 'followers_url': 'https://api.github.com/user/followers',
 'following_url': 'https://api.github.com/user/following{/target}',
 'gists_url': 'https://api.github.com/gists{/gist_id}',
 'hub_url': 'https://api.github.com/hub',
 'issue_search_url': 'https://api.github.com/search/issues?q={query}{&page,per_page,sort,order}',
 'issues_url': 'https://api.github.com/issues',
 'keys_url': '

The type of the return value of .json() is a dictionary, so you can access values in the object by key:

In [14]:
response_dict = response.json()

In [15]:
response_dict["emojis_url"]

'https://api.github.com/emojis'

## View Response Headers

The response headers can give you useful information, such as the content type of the response payload and how long to cache the response. To view these headers, access .headers:

In [16]:
import requests

response = requests.get("https://api.github.com")
response.headers

{'Date': 'Tue, 15 Sep 2026 01:58:05 GMT', 'Cache-Control': 'public, max-age=60, s-maxage=60', 'Vary': 'Accept,Accept-Encoding, Accept, X-Requested-With', 'ETag': '"4f825cc84e1c733059d46e76e6df9db557ae5254f9625dfe8e1b09499c449438"', 'x-github-api-version-selected': '2022-11-28', 'Access-Control-Expose-Headers': 'ETag, Link, Location, Retry-After, X-GitHub-OTP, X-RateLimit-Limit, X-RateLimit-Remaining, X-RateLimit-Used, X-RateLimit-Resource, X-RateLimit-Reset, X-OAuth-Scopes, X-Accepted-OAuth-Scopes, X-Poll-Interval, X-GitHub-Media-Type, X-GitHub-SSO, X-GitHub-Request-Id, Deprecation, Sunset, Warning', 'Access-Control-Allow-Origin': '*', 'Strict-Transport-Security': 'max-age=31536000; includeSubdomains; preload', 'X-Frame-Options': 'deny', 'X-Content-Type-Options': 'nosniff', 'X-XSS-Protection': '0', 'Referrer-Policy': 'origin-when-cross-origin, strict-origin-when-cross-origin', 'Content-Security-Policy': "default-src 'none'", 'Server': 'github.com', 'Content-Type': 'application/json; ch

The .headers attribute returns a dictionary-like object, allowing you to access header values by key. For example, to see the content type of the response payload, you can access "Content-Type":

In [17]:
response.headers["Content-Type"]

'application/json; charset=utf-8'

There’s something special about this dictionary-like headers object. The HTTP specification defines headers as case-insensitive, which means you can access them without worrying about their capitalization:

In [18]:
response.headers["content-type"]

'application/json; charset=utf-8'

Whether you use the key "content-type" or "Content-Type", you’ll get the same value.

Now that you’ve seen the most useful attributes and methods of Response in action, you already have a good overview of Requests’ basic usage. You can get content from the internet and work with the response that you receive.

But there’s more to the internet than plain, straightforward URLs. In the next section, you’ll take a step back and see how your responses change when you customize your GET requests to account for query string parameters.



## Add Query String Parameters

One common way to customize a GET request is to pass values through query string parameters in the URL. To do this using get(), you pass data to params. For example, you can use GitHub’s repository search API to look for popular Python repositories:

In [19]:
import requests

response = requests.get(
    "https://api.github.com/search/repositories",
    params={"q": "language:python", "sort": "stars", "order": "desc"},
)

json_response = response.json()
popular_repositories = json_response["items"]
for repo in popular_repositories[:3]:
    print(f"Name: {repo['name']}")
    print(f"Description: {repo['description']}")
    print(f"Stars: {repo['stargazers_count']}\n")

Name: public-apis
Description: A collective list of free APIs
Stars: 480225

Name: free-programming-books
Description: :books: Freely available programming books
Stars: 396776

Name: system-design-primer
Description: Learn how to design large-scale systems. Prep for the system design interview.  Includes Anki flashcards.
Stars: 370030



By passing a dictionary to the params parameter of get(), you’re able to modify the results that come back from the search API.

You can pass params to get() either as a dictionary, as you’ve just done, or as a list of tuples:

In [20]:
import requests

requests.get(
    "https://api.github.com/search/repositories",
    [("q", "language:python"), ("sort", "stars"), ("order", "desc")],
)

<Response [200]>

You can even pass the values as bytes:

In [21]:
requests.get(
    "https://api.github.com/search/repositories",
    params=b"q=language:python&sort=stars&order=desc",
)

<Response [200]>

## Customize Request Headers 

To customize headers, you pass a dictionary of HTTP headers to get() using the headers parameter. For example, you can change your previous search request to highlight matching search terms in the results by specifying the text-match media type in the Accept header:

In [22]:
import requests

response = requests.get(
    "https://api.github.com/search/repositories",
    params={"q": '"real python"'},
    headers={"Accept": "application/vnd.github.text-match+json"},
)

json_response = response.json()
first_repository = json_response["items"][0]
print(first_repository["text_matches"][0]["matches"])

KeyError: 'items'

## Use Other HTTP Methods

Aside from GET, other popular HTTP methods include POST, PUT, DELETE, HEAD, PATCH, and OPTIONS. For each of these HTTP methods, Requests provides a function with a similar signature to get()

In [33]:
requests.get("https://httpbin.org/get")


<Response [200]>

In [34]:
requests.post("https://httpbin.org/post", data={"key": "value"})

<Response [200]>

In [35]:
requests.put("https://httpbin.org/put", data={"key": "value"})

<Response [200]>

In [36]:
requests.delete("https://httpbin.org/delete")

<Response [200]>

In [37]:
requests.head("https://httpbin.org/get")

<Response [200]>

In [38]:
requests.patch("https://httpbin.org/patch", data={"key": "value"})

<Response [200]>

In [39]:
requests.options("https://httpbin.org/get")

<Response [200]>

In the example above, you called each function to make a request to the httpbin service using the corresponding HTTP method.

All of these functions are high-level shortcuts to requests.request(), which takes the method name as its first argument:

In [40]:
requests.request("GET", "https://httpbin.org/get")

<Response [200]>

In [41]:
import requests
from requests.exceptions import HTTPError

URLS = ["https://api.github.com", "https://api.github.com/invalid"]

for url in URLS:
    try:
        response = requests.get(url)
        response.raise_for_status()
    except HTTPError as http_err:
        print(f"HTTP error occurred: {http_err}")
    except Exception as err:
        print(f"Other error occurred: {err}")
    else:
        print("Success!")

Success!
HTTP error occurred: 404 Client Error: Not Found for url: https://api.github.com/invalid


In [ ]:
python text_matches.py
[{'text': 'Real Python', 'indices': [23, 34]}]

SyntaxError: invalid syntax (162987435.py, line 1)

Use Other HTTP Methods
Aside from GET, other popular HTTP methods include POST, PUT, DELETE, HEAD, PATCH, and OPTIONS. For each of these HTTP methods, Requests provides a function with a similar signature to get().

In [43]:
import requests

requests.get("https://httpbin.org/get")

requests.post("https://httpbin.org/post", data={"key": "value"})

requests.put("https://httpbin.org/put", data={"key": "value"})

requests.delete("https://httpbin.org/delete")

requests.head("https://httpbin.org/get")

requests.patch("https://httpbin.org/patch", data={"key": "value"})

requests.options("https://httpbin.org/get")

<Response [200]>

##Send Request Data

according to the HTTP specification, POST, PUT, and the less common PATCH requests pass their data through the message body rather than through parameters in the query string. With Requests, you pass this payload to the corresponding function’s data parameter.

In [44]:
import requests

requests.post("https://httpbin.org/post", data={"key": "value"})

<Response [200]>

In [45]:
requests.post("https://httpbin.org/post", data=[("key", "value")])

<Response [200]>

In [46]:
response = requests.post("https://httpbin.org/post", json={"key": "value"})
json_response = response.json()
json_response["data"]

json_response["headers"]["Content-Type"]

'application/json'

You can see from the response that the server received your request data and headers just as you sent them. Requests also provides this information to you in the form of a PreparedRequest, which you’ll look at more closely in the next section.



##Inspect the Prepared Request

When you make a request, the Requests library prepares the request before actually sending it to the destination server. Request preparation includes things like validating headers and serializing JSON content.

You can view the PreparedRequest object by accessing .request on a Response object:

In [47]:
import requests

response = requests.post("https://httpbin.org/post", json={"key":"value"})

response.request


response.request.headers["Content-Type"]


response.request.url


response.request.body

b'{"key": "value"}'

##Use Authentication

Authentication helps a service understand who you are. Typically, you provide your credentials to a server by passing data through the Authorization header or a custom header defined by the service. All the functions in Requests that you’ve seen to this point provide a parameter called auth, which allows you to pass your credentials directly:


In [48]:
import requests

response = requests.get(
    "https://httpbin.org/basic-auth/user/passwd",
    auth=("user", "passwd")
)

response.status_code

response.request.headers["Authorization"]

'Basic dXNlcjpwYXNzd2Q='

You could make the same request by passing explicit basic authentication credentials using HTTPBasicAuth:

In [49]:
from requests.auth import HTTPBasicAuth
requests.get(
    "https://httpbin.org/basic-auth/user/passwd",
    auth=HTTPBasicAuth("user", "passwd")
)

<Response [200]>

Although you don’t need to be explicit for basic authentication, you might want to authenticate using another method. Requests provides other methods of authentication out of the box, such as HTTPDigestAuth and HTTPProxyAuth.

A real-world example of an API that requires authentication is GitHub’s authenticated user API. This endpoint provides information about the authenticated user’s profile.

If you try to make a request without credentials, then you’ll see that the status code is 401 Unauthorized:

In [50]:
requests.get("https://api.github.com/user")

<Response [401]>

In [51]:
import requests

token = "<YOUR_GITHUB_PA_TOKEN>"
response = requests.get(
    "https://api.github.com/user",
    auth=("", token)
)
response.status_code

401

As you learned earlier, this approach passes the credentials to HTTPBasicAuth, which expects a username and a password, and sends the credentials as a Base64-encoded string with the prefix "Basic ":



In [52]:
response.request.headers["Authorization"]

'Basic OjxZT1VSX0dJVEhVQl9QQV9UT0tFTj4='

This method works, but it’s not the right way to authenticate with a Bearer token—and using an empty string input for the superfluous username is awkward.

With Requests, you can supply your own authentication mechanism to fix that. To try this out, create a subclass of AuthBase and implement .__call__():

In [53]:
from requests.auth import AuthBase

class TokenAuth(AuthBase):
    """Implements a token authentication scheme."""

    def __init__(self, token):
        self.token = token

    def __call__(self, request):
        """Attach an API token to the Authorization header."""
        request.headers["Authorization"] = f"Bearer {self.token}"
        return request

Here, your custom TokenAuth mechanism receives a token, then includes that token in the Authorization header of your request, also setting the recommended "Bearer " prefix to the string.

You can now use this custom token authentication to make your call to GitHub’s authenticated user API:

In [54]:
import requests
from custom_token_auth import TokenAuth

token = "<YOUR_GITHUB_PA_TOKEN>"
response = requests.get(
    "https://api.github.com/user",
    auth=TokenAuth(token)
)

response.status_code

response.request.headers["Authorization"]

ModuleNotFoundError: No module named 'custom_token_auth'

##Communicate Securely With Servers

Whenever the data you’re trying to send or receive is sensitive, security becomes essential. The way that you communicate with secure sites over HTTP is by establishing an encrypted connection using Transport Layer Security (TLS). TLS is the successor to Secure Sockets Layer (SSL), offering enhanced security and efficiency in secure communications. Yet, it’s still common for programmers to use the term SSL instead of TLS.

For example, when you’re working in a corporate environment with custom certificate authorities, you may need to provide your own certificate bundle:

In [55]:
import requests

requests.get(
    "https://internal-api.company.com",
    verify="/path/to/company-ca.pem"
)

OSError: Could not find a suitable TLS CA certificate bundle, invalid path: /path/to/company-ca.pem

If you’re debugging certificate issues in development, then you might be tempted to disable verification entirely. While this works, it’s a significant security risk and you should never use this approach in production:

In [56]:
requests.get("https://api.github.com", verify=False)

/Users/anisarodriguez/Desktop/CS315/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.github.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


<Response [200]>

Requests warns you about this dangerous practice because disabling verification makes you vulnerable to man-in-the-middle attacks.

## improve performance

When using Requests, especially in a production application environment, it’s important to consider performance implications. Features like timeout control, sessions, and retry limits can help you keep your application running smoothly.

## Set Request Timeouts

When you make an inline request to an external service, your system must wait for the response before moving on. If your application waits too long for that response, requests to your service could back up, your user experience could suffer, or background jobs might hang.

By default, Requests will wait indefinitely on the response, so you should almost always specify a timeout duration to prevent these issues from happening. To set the request’s timeout, use the timeout parameter. timeout can be an integer or float representing the number of seconds to wait on a response before timing out:

In [57]:
requests.get("https://api.github.com", timeout=1)


requests.get("https://api.github.com", timeout=0.01)

ConnectTimeout: HTTPSConnectionPool(host='api.github.com', port=443): Max retries exceeded with url: / (Caused by ConnectTimeoutError(<HTTPSConnection(host='api.github.com', port=443) at 0x10f0e1bd0>, 'Connection to api.github.com timed out. (connect timeout=1)'))

You can also pass a tuple to timeout with the following two elements:

Connect timeout: The amount of time it allows the client to establish a connection to the server
Read timeout: The time it’ll wait for a response once the client has established a connection
Both of these elements should be numbers, and can be of type int or float:



In [58]:
requests.get("https://api.github.com", timeout=(3.05, 5))

<Response [200]>

In [59]:
import requests
from requests.exceptions import Timeout

try:
    response = requests.get("https://api.github.com", timeout=(3.05, 5))
except Timeout:
    print("The request timed out")
else:
    print("The request did not time out")

The request did not time out


## Reuse Connections With Session Objects

Until now, you’ve been dealing with high-level requests APIs such as get() and post(). These functions are abstractions of what’s going on when you make your requests. They hide implementation details, such as how connections are managed, so you don’t have to worry about them.

In [60]:
import requests
from custom_token_auth import TokenAuth

TOKEN = "<YOUR_GITHUB_PA_TOKEN>"

with requests.Session() as session:
    session.auth = TokenAuth(TOKEN)

    first_response = session.get("https://api.github.com/user")
    second_response = session.get("https://api.github.com/user")

print(first_response.headers)
print(second_response.json())

ModuleNotFoundError: No module named 'custom_token_auth'

## Retry Failed Requests

When a request fails, you might want your application to retry the same request. However, Requests won’t do this for you by default. To apply this functionality, you need to implement a custom transport adapter.

Transport adapters let you define a set of configurations for each service you interact with. For example, say that you want all requests to https://api.github.com to retry twice before finally raising a RetryError. In that case, you’d build a transport adapter, set its max_retries parameter, and mount it to an existing Session:

In [61]:
import requests
from requests.adapters import HTTPAdapter
from requests.exceptions import RetryError
from urllib3.util.retry import Retry

retry_strategy = Retry(
    total=2,
    status_forcelist=[429, 500, 502, 503, 504]
)
github_adapter = HTTPAdapter(max_retries=retry_strategy)

with requests.Session() as session:
    session.mount("https://api.github.com", github_adapter)
    try:
        response = session.get("https://api.github.com/")
    except RetryError as err:
        print(f"Error: {err}")